# Feature Engineering

Build and validate candidate features using only information available at prediction time.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)

TRAIN_PATH = "../data/processed/train_relationship_features.csv"
TEST_PATH = "../data/processed/test_relationship_features.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test:", test.shape)

display(train.head())

Train: (1000, 26)
Test: (350, 20)


,Test_ID,Applied_Voltage_kV,Load_Current_A,Ambient_Temperature_C,Test_Duration_min,Sensor_S1,Sensor_S2,Sensor_S3,Sensor_S4,Reference_Parameter,Validity_Label,S2_S1_residual,S3_S1_residual,S3_S2_residual,Reference_Current_residual,S2_S1_residual_abs,S3_S1_residual_abs,S3_S2_residual_abs,S2_minus_S1,S3_minus_S1,S3_minus_S2,Voltage_Current,Sensor_S1_missing,Sensor_S2_missing,Sensor_S3_missing,Sensor_S4_missing
0,TRN-0889,16.7196,93.1228,33.3421,19.9546,13.6343,15.1361,16.5062,62.9115,34.8501,Valid,1.388878,-0.614474,-2.390665,-2.059938,1.388878,0.614474,2.390665,1.5018,2.8719,1.3701,1556.975967,0,0,0,0
1,TRN-0820,21.4477,82.5230,34.4915,47.0553,15.6466,15.9849,19.3289,39.8667,30.4762,Valid,0.510395,-0.358130,-0.632677,0.320470,0.510395,0.358130,0.632677,0.3383,3.6823,3.3440,1769.928547,0,0,0,0
2,TRN-0411,19.2432,107.3619,29.4673,5.2763,15.1080,16.6481,18.3834,44.5390,46.8046,Valid,1.634430,-0.615597,-2.409982,-2.775351,1.634430,0.615597,2.409982,1.5401,3.2754,1.7353,2065.986514,0,0,0,0
3,TRN-0754,22.7191,69.9690,26.9695,20.3051,14.9296,14.7026,18.6269,44.2673,23.7153,Valid,-0.158190,-0.144383,0.273858,-0.853925,0.158190,0.144383,0.273858,-0.2270,3.6973,3.9243,1589.632708,0,0,0,0
4,TRN-0707,13.5008,108.8999,35.5415,29.1939,12.6845,15.2455,14.6132,44.0720,48.5994,Valid,2.318794,-1.300196,-4.420901,-2.627775,2.318794,1.300196,4.420901,2.5610,1.9287,-0.6323,1470.235770,0,0,0,0


In [5]:
CLASS_TARGET = "Validity_Label"
REG_TARGET = "Reference_Parameter"

# Features that exist in BOTH train and test
feature_cols = [
    col for col in train.columns
    if col in test.columns
    and col not in [
        "Test_ID",
        CLASS_TARGET,
        REG_TARGET
    ]
]

print("Number of model features:", len(feature_cols))
print("\nFeatures:")
for col in feature_cols:
    print("-", col)

Number of model features: 19

Features:
- Applied_Voltage_kV
- Load_Current_A
- Ambient_Temperature_C
- Test_Duration_min
- Sensor_S1
- Sensor_S2
- Sensor_S3
- Sensor_S4
- S2_S1_residual
- S3_S1_residual
- S3_S2_residual
- S2_minus_S1
- S3_minus_S1
- S3_minus_S2
- Voltage_Current
- Sensor_S1_missing
- Sensor_S2_missing
- Sensor_S3_missing
- Sensor_S4_missing


In [6]:
train_only = set(train.columns) - set(test.columns)

print("\nColumns only in TRAIN:")
print(train_only)


Columns only in TRAIN:
{'S3_S1_residual_abs', 'Reference_Current_residual', 'S2_S1_residual_abs', 'Validity_Label', 'Reference_Parameter', 'S3_S2_residual_abs'}


In [7]:
from sklearn.impute import SimpleImputer

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_class = train[CLASS_TARGET]
y_reg = train[REG_TARGET]

print("X:", X.shape)
print("X_test:", X_test.shape)

X: (1000, 19)
X_test: (350, 19)


In [8]:
imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("After imputation:")
print("X:", X_imputed.shape)
print("X_test:", X_test_imputed.shape)

print("Remaining missing in X:", X_imputed.isna().sum().sum())
print("Remaining missing in X_test:", X_test_imputed.isna().sum().sum())

After imputation:
X: (1000, 19)
X_test: (350, 19)
Remaining missing in X: 0
Remaining missing in X_test: 0
